<span style="color:red">If you only want to test the currently published package, you can skip the DEVELOPER VERSION editable install step.</span>

<span style="color:red">DEVELOPER VERSION</span>

If you want to test new allocation methods that are still being implemented, use the in repo code.
Install the package in editable mode from the repository root, then restart the kernel before running the notebook.

```powershell/cmd
python -m pip install -e .
```

# Allocation Validation Runner

This notebook runs the allocation validation workflow and prints report paths + pass summary.

Use the config cells to select different arguments before running the validation.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for parent in (start, *start.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
from pyaesa import set_workspace

# Windows example; update this path before running.
set_workspace(r"C:\Users\username\Documents\aesa_workspace")
# macOS example; update this path before running.
# set_workspace("/Users/username/Documents/aesa_workspace")

## 1) Load Validation Utilities and Data

In [ ]:
import importlib

import pandas as pd

# REPO_ROOT is configured in the setup cell above.
# Import the validation module.
validation_module = importlib.import_module(
    "tests.allocation_equation_validation.validation_function.test_alloc_methods")

## 2) Load Config Options for Validation

Edit these values if needed before running validation.

### Config explanation


- `REFRESH_ALLOCATE_OUTPUTS`: If `True`, force rerun `deterministic_asocc` for each source/FU/L1-mode target. If `False`, reuse compatible complete outputs and fail fast on incompatible targets.
- `REFRESH_VALIDATION_REPORTS`: If `True`, recompute validation CSVs. If `False`, reuse existing validation CSVs when present.

- `YEAR`: Single year to validate (e.g., `2019`).
- `REFERENCE_YEARS`: AR reference year(s); `None` uses all available historical years.
- `PROJECT_NAME`: Validation root name. The runner writes validation CSVs to `<PROJECT_NAME>/validation` and deterministic aSoCC outputs to per-target projects derived from this root.
- `SOURCES`: MRIO source(s) (e.g., `"exiobase_3102_ixi"` or list).
- `L1_FUS`: L1 functional units to include (e.g., `("L1.a", "L1.b")`).
- `L2_FUS`: L2 functional units to include.
- `EXIO_LCIA_METHODS`: LCIA method name(s) for exiobase runs.
- `L2_BUCKETS`: L2 output buckets checked. Use `("l2_vs_global",)` for final public outputs only, or include `"l2_in_l1"` when validating two step support rows.
- `ATOL`: Absolute tolerance for checks (sum to-1 for `L1.*` and non-`b` `L2.*`; overlap equality checks for `L2*b`).
- `OUTPUT_FORMAT`: deterministic aSoCC output format to produce in the run (`"csv"`, `"pickle"`, or `"parquet"`).
- `GROUP_REG`, `GROUP_SEC`: Whether to use aggregated regions/sectors.
- `AGG_VERSION`: Aggregation version tag (must match your aggregation CSVs).
- `L1_REG_AGGREG`: L1 aggregation order (`"pre"`, `"post"`, or `"both"`).
- `METHOD_PLAN`: Method selection plan (`"default"`, `"one_step"`, `"two_steps"`, `"pairs"`, `"one_step_pairs"`).
- `L1_METHODS`: Optional subset of L1 methods.
- `ONE_STEP_METHODS`: Optional subset of one step L2 methods.
- `TWO_STEP_METHODS`: Optional subset of two step L2 methods.
- `L1_L2_PAIRS`: Explicit `L1::L2` pairs (used only in `pairs`/`one_step_pairs`).

### Config settings

In [ ]:
# Configure your run here

REFRESH_ALLOCATE_OUTPUTS = False
REFRESH_VALIDATION_REPORTS = True

YEAR = 2019
REFERENCE_YEARS = 1995
PROJECT_NAME = "allocation_validation"
SOURCES = "exiobase_3102_ixi"
L1_FUS = ("L1.a", "L1.b")
L2_FUS = ("L2.a.a", "L2.a.b", "L2.a.c", "L2.b.a", "L2.b.b", "L2.c.a", "L2.c.b")
EXIO_LCIA_METHODS = "gwp100_lcia"
L2_BUCKETS = ("l2_in_l1", "l2_vs_global")
ATOL = 1e-6
OUTPUT_FORMAT = "csv"

GROUP_REG = None
GROUP_SEC = None
AGG_VERSION = None
L1_REG_AGGREG = "pre"

METHOD_PLAN = "default"
L1_METHODS = None
ONE_STEP_METHODS = None
TWO_STEP_METHODS = None
L1_L2_PAIRS = None

## 3) Run Validation Function

### Validation Rules (By FU and Method Type)


- `L1.*`: world level deterministic check; for each `(method, impact, reference_year)` aggregate, sum of shares must be `1` (within `ATOL`).

- `L2.*` non-`b` FUs (`L2.a.a`, `L2.a.c`, `L2.b.a`, `L2.c.a`):
  - `l2_in_l1`: deterministic check to `1` by FU country axis (`r_f` for `L2.a.a`/`L2.b.a`/`L2.c.a`, `r_p` for `L2.a.c`).
  - `l2_vs_global`: deterministic world level check to `1`.
  - LCIA rows: add `F_Y` share before checking (`sum_share_observed + fy_add_share_observed`).

- `L2*b` FUs (`L2.a.b`, `L2.b.b`, `L2.c.b`):
  - Deterministic overlap check (not universal sum to-1): observed overlap (`all_incl_fy_share_observed`) is compared to method specific expected overlap.
  - `l2_in_l1` is validated for `UT(FDa)` and `UT(GVAa)` only:
    - `UT(FDa)`: per `r_f`, expected overlap uses adjusted utility propagation numerator `sum(x_to_rc * kappa)` divided by `FD_r_f`.
    - `UT(GVAa)`: per `r_u`, expected overlap uses adjusted numerator `sum(x * omega_reg)` divided by `GVA_r_u`.
  - `l2_vs_global`:
    - one step `UT(TD)`: expected `X_W / FD_W`.
    - combined `UT(FDa)` / `UT(GVAa)`: expected is L1-weighted global overlap from the corresponding per region adjusted overlaps.
    - one step `AR(E^{CBA_TD})`: expected by impact is `(E_CBA_TD_world + F_Y_world) / E_CBA_FD_world`.

  - Process level assertions already enforced in `process_mrio` for utility propagation consistency:
    - `x_to_rc` row totals match `x` (per `(r_p, s_p)`).
    - `kappa` rows sum to `1` for nonzero rows (and `0` for zero rows).
    - `omega_reg` columns sum to `1` for nonzero columns (and `0` for zero columns).
    - See the methodological appendix for detailed properties of `kappa`/`omega` and the utility propagation interpretation behind `UT(FDa)` and `UT(GVAa)`.

- Report interpretation columns:
  - Core observed values: `sum_share_observed`, `fy_add_share_observed`, `all_incl_fy_share_observed`.
  - Deterministic target/diagnostics: `ratio_expected`, `abs_error`, `atol_used`.
  - Rows identification: `group_key`, `l2_country_axis`, `l2_country_code`.


### Validation function

In [ ]:
config = {
    "YEAR": YEAR,
    "REFERENCE_YEARS": REFERENCE_YEARS,
    "PROJECT_NAME": PROJECT_NAME,
    "SOURCES": SOURCES,
    "L1_FUS": L1_FUS,
    "L2_FUS": L2_FUS,
    "EXIO_LCIA_METHODS": EXIO_LCIA_METHODS,
    "L2_BUCKETS": L2_BUCKETS,
    "ATOL": ATOL,
    "OUTPUT_FORMAT": OUTPUT_FORMAT,
    "GROUP_REG": GROUP_REG,
    "GROUP_SEC": GROUP_SEC,
    "AGG_VERSION": AGG_VERSION,
    "L1_REG_AGGREG": L1_REG_AGGREG,
    "METHOD_PLAN": METHOD_PLAN,
    "L1_METHODS": L1_METHODS,
    "ONE_STEP_METHODS": ONE_STEP_METHODS,
    "TWO_STEP_METHODS": TWO_STEP_METHODS,
    "L1_L2_PAIRS": L1_L2_PAIRS,
    "REFRESH_ALLOCATE_OUTPUTS": REFRESH_ALLOCATE_OUTPUTS,
    "REFRESH_VALIDATION_REPORTS": REFRESH_VALIDATION_REPORTS,
}

validation_module.run_allocation_methods_with_config(
    config)